In [1]:
from sentence_transformers import CrossEncoder
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import json
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [2]:
import json
import pandas as pd

DEV_PATH = "../Data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl"

rows = []
with open(DEV_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

df = pd.DataFrame(rows)
df.head()


,anchor_text,text_a,text_b,text_a_is_closer
0,The book follows an international organization...,The old grandmother Tina arrives in town to at...,The nano-plague that poisoned Earth's water su...,False
1,"Glenn Tyler (Elvis Presley), a childish 25-yea...","Bill Babbitt supported the death penalty, unti...",A white-collar suburban father Kyle (Fran Kran...,True
2,Signaller Charles Plumpick (Bates) is a kilt-w...,"Sid, Russ and Jerry are three wannabe criminal...",Brendan Byers III is a rich playboy who enlist...,False
3,Barbara is married to the distinguished profes...,Eddie Quinn's unruly wife Maureen drinks and s...,Jerome Littlefield is an orderly at a hospital...,False
4,A wealthy widower locks up his two grown-up ch...,Barbara is married to the distinguished profes...,Stefano (Lino Capolicchio) arrives in a villag...,False


In [3]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
len(train_df), len(val_df)


(160, 40)

In [4]:
train_pairs = []
train_labels = []

for _, row in train_df.iterrows():
    anchor = row["anchor_text"]
    A = row["text_a"]
    B = row["text_b"]
    label_A = bool(row["text_a_is_closer"])

    # (anchor, A)
    train_pairs.append([anchor, A])
    train_labels.append(1 if label_A else 0)

    # (anchor, B)
    train_pairs.append([anchor, B])
    train_labels.append(0 if label_A else 1)

len(train_pairs), len(train_labels)


(320, 320)

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader

class PairwiseHF(Dataset):
    def __init__(self, pairs, labels, tokenizer):
        self.pairs = pairs
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        text1, text2 = self.pairs[idx]
        enc = self.tokenizer(
            text1,
            text2,
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}
        enc["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return enc


In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "roberta-large"


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
train_dataset = PairwiseHF(train_pairs, train_labels, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

In [8]:
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def predict(anchor, A, B):
    """Return: pred(A/B), margin, score_A, score_B"""
    def score(cont):
        enc = tokenizer(anchor, cont, return_tensors="pt",
                        truncation=True, padding=True).to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        prob = F.softmax(logits, dim=-1)[0][1].item()
        return prob

    score_A = score(A)
    score_B = score(B)
    pred = "A" if score_A > score_B else "B"
    margin = abs(score_A - score_B)
    return pred, margin, score_A, score_B


In [9]:
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm
import numpy as np

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
epochs = 5

best_val_acc = 0.0
best_state = None

for epoch in range(epochs):
    print(f"\n================ Epoch {epoch+1}/{epochs} ================")
    model.train()

    loop = tqdm(train_loader, desc="Training")
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        loop.set_postfix(loss=loss.item())

    # ---------------- Validation ----------------
    print("Validating...")
    model.eval()

    val_preds = []
    val_golds = []
    val_margins = []

    for _, row in val_df.iterrows():
        anchor = row["anchor_text"]
        A = row["text_a"]
        B = row["text_b"]
        gold = "A" if row["text_a_is_closer"] else "B"

        pred, margin, _, _ = predict(anchor, A, B)
        val_preds.append(pred)
        val_golds.append(gold)
        val_margins.append(margin)

    val_acc = accuracy_score(val_golds, val_preds)
    val_margin = float(np.mean(val_margins))

    print(f"Epoch {epoch+1} | val_acc={val_acc:.4f} | avg_margin={val_margin:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print("  -> New best model!")

# Restore best checkpoint
if best_state is not None:
    model.load_state_dict(best_state)

print("\nBest validation accuracy:", best_val_acc)



================ Epoch 1/5 ================


Training:   0%|          | 0/80 [00:00<?, ?it/s]

Validating...
Epoch 1 | val_acc=0.6250 | avg_margin=0.0004
  -> New best model!

================ Epoch 2/5 ================


Training:   0%|          | 0/80 [00:00<?, ?it/s]

Validating...
Epoch 2 | val_acc=0.4750 | avg_margin=0.0004

================ Epoch 3/5 ================


...

Validating...
Epoch 3 | val_acc=0.6000 | avg_margin=0.0004

================ Epoch 4/5 ================


Training:   0%|          | 0/80 [00:00<?, ?it/s]

Validating...
Epoch 4 | val_acc=0.5750 | avg_margin=0.0017

================ Epoch 5/5 ================


Training:   0%|          | 0/80 [00:00<?, ?it/s]

Validating...
Epoch 5 | val_acc=0.5250 | avg_margin=0.0008

Best validation accuracy: 0.625
